## Predicting Gender using Multinomial Naive Bayes

This notebook demonstrates how to predict gender based on occupation, eye color, and nationality using a Multinomial Naive Bayes model. We will cover data loading, cleaning, transformation, model training, evaluation, and making a specific prediction.

### Step 1: Load the Dataset

First, we will load the `gender_prdct.csv` file into a pandas DataFrame to begin our analysis.

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/gender_prdct.csv')

# Display the first few rows of the DataFrame
print("Original DataFrame head:")
display(df.head())

# Display basic information about the dataset
print("\nDataFrame Info:")
df.info()

Original DataFrame head:


,Occupation,Eye Color,Nationality,Gender
0,actuary,green,korea,F
1,barista,green,italy,M
2,dentist,hazel,japan,M
3,dentist,green,italy,F
4,chemist,hazel,japan,M



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Occupation   14 non-null     object
 1   Eye Color    14 non-null     object
 2   Nationality  14 non-null     object
 3   Gender       14 non-null     object
dtypes: object(4)
memory usage: 580.0+ bytes


### Step 2: Data Cleaning and Transformation

We need to prepare the data for the Multinomial Naive Bayes model. This involves:
1.  **Handling Missing Values**: Check for any missing values in the dataset and decide on an appropriate strategy (e.g., dropping rows, imputation).
2.  **Categorical Feature Encoding**: Convert the categorical features (`Occupation`, `Eye Color`, `Nationality`) into numerical format using one-hot encoding, as required by the model.
3.  **Target Variable Encoding**: Convert the target variable `Gender` into numerical labels.
4.  **Feature and Target Separation**: Separate the features (X) from the target variable (y).

In [2]:
from sklearn.preprocessing import LabelEncoder

# 1. Handle Missing Values
print("\nMissing values before handling:")
display(df.isnull().sum())

# For this dataset, we'll drop rows with any missing values, if any exist.
df.dropna(inplace=True)
print("\nMissing values after dropping rows (if any):")
display(df.isnull().sum())

# 2. Categorical Feature Encoding using One-Hot Encoding
# Store original column names for later use in prediction
categorical_features = ['Occupation', 'Eye Color', 'Nationality']
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=False)

# 3. Target Variable Encoding
le = LabelEncoder()
df_encoded['Gender'] = le.fit_transform(df_encoded['Gender'])

# Store the label encoder for inverse transformation during prediction
gender_label_encoder = le

# 4. Feature and Target Separation
X = df_encoded.drop('Gender', axis=1)
y = df_encoded['Gender']

print("\nTransformed DataFrame head:")
display(df_encoded.head())
print("\nFeatures (X) shape:", X.shape)
print("Target (y) shape:", y.shape)


Missing values before handling:


,0
Occupation,0
Eye Color,0
Nationality,0
Gender,0



Missing values after dropping rows (if any):


,0
Occupation,0
Eye Color,0
Nationality,0
Gender,0



Transformed DataFrame head:


,Gender,Occupation_actuary,Occupation_barista,Occupation_chemist,Occupation_dentist,Eye Color_ green,Eye Color_ hazel,Nationality_ italy,Nationality_ japan,Nationality_ korea
0,0,True,False,False,False,True,False,False,False,True
1,1,False,True,False,False,True,False,True,False,False
2,1,False,False,False,True,False,True,False,True,False
3,0,False,False,False,True,True,False,True,False,False
4,1,False,False,True,False,False,True,False,True,False



Features (X) shape: (14, 9)
Target (y) shape: (14,)


### Step 3: Model Training

Now, we will split the dataset into training and testing sets and then train a Multinomial Naive Bayes classifier on the training data.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and train the Multinomial Naive Bayes model
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully.")

Multinomial Naive Bayes model trained successfully.


### Step 4: Model Evaluation

We will evaluate the trained model's performance by calculating its accuracy on the test set.

In [4]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on the test set
y_pred = mnb.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on the test set: {accuracy:.4f}")

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=gender_label_encoder.classes_))

Model Accuracy on the test set: 0.6667

Classification Report:
              precision    recall  f1-score   support

           F       1.00      0.50      0.67         2
           M       0.50      1.00      0.67         1

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3



### Step 5: Predict for a Specific Case

Finally, we will use the trained model to predict the gender for the given case: `[Occupation = actuary, Eye Color = green, Nationality = Japan]`.

In [5]:
# Create a DataFrame for the new case
new_case_data = {
    'Occupation': ['actuary'],
    'Eye Color': ['green'],
    'Nationality': ['Japan']
}
new_case_df = pd.DataFrame(new_case_data)

# Apply the same one-hot encoding as the training data
# We need to ensure the new case DataFrame has the exact same columns as X (training features)
# Use reindex to add missing columns and fill with 0, and drop extra columns
new_case_encoded = pd.get_dummies(new_case_df, columns=categorical_features, drop_first=False)

# Align columns with the training data's columns
new_case_processed = new_case_encoded.reindex(columns=X.columns, fill_value=0)

# Predict the gender for the new case
predicted_gender_encoded = mnb.predict(new_case_processed)

# Inverse transform the predicted gender to get the original label
predicted_gender = gender_label_encoder.inverse_transform(predicted_gender_encoded)

print(f"Predicted Gender for [Occupation = actuary, Eye Color = green, Nationality = Japan]: {predicted_gender[0]}")

Predicted Gender for [Occupation = actuary, Eye Color = green, Nationality = Japan]: F
